In [9]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import chirp, correlate
from scipy import signal
from scipy import constants
from scipy.signal import butter, filtfilt

def low_pass_filter(cutoff_freq, fs, order, data):
    # Design the low-pass filter
    nyq = 0.5 * fs
    normal_cutoff = cutoff_freq / nyq
    b, a = butter(order, normal_cutoff, btype='lowpass')

    # Apply the filter
    filtered_data = filtfilt(b, a, data)
    return filtered_data

def process_raw_samples(fileName_rx, fileName_ref, burst_len, num_steps,transition_width, cut_off, resampling_factor=1):
    ref_samples = np.fromfile(fileName_ref, np.complex64)
    rx_samples = np.fromfile(fileName_rx, np.complex64) 

    #normalise samples
    ref_samples = low_pass_filter(transition_width,cut_off,4,ref_samples)
    rx_samples = low_pass_filter(transition_width,cut_off,4,rx_samples)
    # ref_samples = normaliseSignals(ref_samples,burst_len,num_steps)
    # rx_samples = normaliseSignals(rx_samples,burst_len,num_steps)

    #reshape samples into busrt
    rx_samples_reshaped = rx_samples.reshape([num_steps,-1])
    #rx_samples_reshaped = signal.resample(rx_samples_reshaped,int(burst_len*resampling_factor),axis=1)
    #rx_samples_reshaped = signal.resample_poly(rx_samples_reshaped,resampling_factor,1,axis=1)
    ref_samples_reshaped = ref_samples.reshape([num_steps,-1])
    #ref_samples_reshaped = signal.resample(ref_samples_reshaped,int(burst_len*resampling_factor),axis=1)
    #ref_samples_reshaped = signal.resample_poly(ref_samples_reshaped,resampling_factor,1,axis=1)

    #drop noisy samples
    # rx_samples_reshaped = rx_samples_reshaped[:,100:100+int(burst_len/2)]
    # ref_samples_reshaped = ref_samples_reshaped[:,100:100+int(burst_len/2)]

    return rx_samples_reshaped, ref_samples_reshaped, rx_samples, ref_samples

def generate_ascan_from_cw(
    echo_signals: np.ndarray,
    reference_signals: np.ndarray,
    num_steps: int,
    step_length: int,
    baseband_freq_hz: float,
    sampling_rate_hz: float
) -> np.ndarray:
    """
    Generates an A-scan range profile for a single-tone CW SFCW radar.

    This function performs phase coherence compensation by dividing the echo
    signal by a reference signal in the frequency domain and then applies an
    IFFT to synthesize the time-domain range profile.

    Args:
        echo_signals: A 1D numpy array containing the concatenated raw IQ
                      samples from the target echo channel.
        reference_signals: A 1D numpy array containing the concatenated raw IQ
                           samples from the reference (loopback) channel.
        num_steps: The total number of frequency steps in the sweep.
        step_length: The number of IQ samples recorded for each frequency step.
        baseband_freq_hz: The known frequency of the CW tone at baseband (in Hz).
        sampling_rate_hz: The sampling rate of the SDR's ADC (in Hz).

    Returns:
        A 1D numpy array representing the complex A-scan (range profile).
    """
    # Reshape the 1D arrays into 2D arrays of (num_steps, step_length)
    echo_matrix = echo_signals.reshape(num_steps, step_length)
    ref_matrix = reference_signals.reshape(num_steps, step_length)

    # Calculate the index of the FFT bin corresponding to the baseband frequency
    baseband_bin = int(baseband_freq_hz / sampling_rate_hz * step_length)

    # --- Step 1: Extract Complex Channel Response for Each Frequency Step ---
    # Perform FFT on each step's data (row-wise)
    echo_fft = np.fft.fft(echo_matrix, axis=1)
    ref_fft = np.fft.fft(ref_matrix, axis=1)

    # Extract the complex amplitude at the baseband frequency bin for all steps
    echo_response = echo_fft[:, baseband_bin]
    ref_response = ref_fft[:, baseband_bin]

    # --- Step 2: Perform Phase Coherence Compensation ---
    # Divide the echo response by the reference response to cancel phase error
    # Add a small epsilon to the denominator to avoid division by zero
    epsilon = 1e-12
    #calibrated_frequency_response = echo_response / (ref_response + epsilon)
    calibrated_frequency_response = np.mean(echo_matrix/ref_matrix,axis=1)
    hamming_window = np.hamming(num_steps)
    windowed_response = calibrated_frequency_response * hamming_window

    # --- Step 3: Synthesize the A-scan (Range Profile) ---
    # Apply an IFFT to the coherent frequency response vector
    # np.fft.ifftshift is used to center the main peak in the resulting profile
    #a_scan = np.fft.ifftshift(np.fft.ifft(calibrated_frequency_response))
    a_scan = np.fft.ifft(windowed_response)

    return a_scan, calibrated_frequency_response, echo_response

def generate_ascan_from_lfm(
    echo_signals: np.ndarray,
    reference_signals: np.ndarray,
    num_steps: int,
    step_length: int,
    chirp_bandwidth: float,
    sampling_rate_hz: float
) -> np.ndarray:
    """
    Generates an A-scan range profile for an LFM-chirp SFCW radar.

    This function performs matched filtering, phase coherence compensation, and
    then applies an IFFT to synthesize the time-domain range profile.

    Args:
        echo_signals: A 1D numpy array containing the concatenated raw IQ
                      samples from the target echo channel.
        reference_signals: A 1D numpy array containing the concatenated raw IQ
                           samples from the reference (loopback) channel.
        num_steps: The total number of frequency steps in the sweep.
        step_length: The number of IQ samples recorded for each frequency step.
        chirp_bandwidth: The bandwidth of the LFM chirp sub-pulse (in Hz).
        sampling_rate_hz: The sampling rate of the SDR's ADC (in Hz).

    Returns:
        A 1D numpy array representing the complex A-scan (range profile).
    """
    # Reshape the 1D arrays into 2D arrays of (num_steps, step_length)
    echo_matrix = echo_signals.reshape(num_steps, step_length)
    ref_matrix = reference_signals.reshape(num_steps, step_length)

    # --- Step 1: Generate the local matched filter (reference chirp) ---
    pulse_duration = step_length / sampling_rate_hz
    t = np.linspace(0, pulse_duration, step_length, endpoint=False)
    local_chirp = chirp(t, f0=-chirp_bandwidth/2, f1=chirp_bandwidth/2, t1=pulse_duration, method='linear')
    
    # --- Step 2: Extract Complex Channel Response via Matched Filtering ---
    calibrated_frequency_response = np.zeros(num_steps, dtype=np.complex64)*np.hamming(num_steps)
    uncalibrated_frequency_response = np.zeros(num_steps, dtype=np.complex64)*np.hamming(num_steps)

    for i in range(num_steps):
        # # Correlate echo with ref to find the compressed peak
        # cor = correlate(echo_matrix[i, :], ref_matrix[i, :], mode='same')
        # #cor = np.fft.fft(echo_matrix[i, :])*np.fft.fft(np.conj(np.flip(ref_matrix[i, :])))
        # peak_index_cor = np.argmax(np.abs(cor))
        # calibrated_frequency_response[i] = cor[peak_index_cor]

        # # Correlate reference with local chirp to find the compressed peak
        # cor = correlate(echo_matrix[i, :], local_chirp, mode='same')
        # #cor = np.fft.fft(echo_matrix[i, :])*np.fft.fft(np.conj(np.flip(local_chirp)))
        # peak_index_cor = np.argmax(np.abs(cor))
        # uncalibrated_frequency_response[i] = cor[peak_index_cor]
        
        # Correlate echo with ref to find the compressed peak
        cor = correlate(echo_matrix[i, :], local_chirp, mode='same')
        #cor = np.fft.fft(echo_matrix[i, :])*np.fft.fft(np.conj(np.flip(ref_matrix[i, :])))
        peak_index_cor = np.argmax(np.abs(cor))
        calibrated_frequency_response[i] = cor[peak_index_cor]

        # Correlate reference with local chirp to find the compressed peak
        cor = correlate(ref_matrix[i, :], local_chirp, mode='same')
        #cor = np.fft.fft(echo_matrix[i, :])*np.fft.fft(np.conj(np.flip(local_chirp)))
        peak_index_cor = np.argmax(np.abs(cor))
        uncalibrated_frequency_response[i] = cor[peak_index_cor]

    # # --- Step 3: Perform Phase Coherence Compensation ---
    epsilon = 1e-12
    calibrated_frequency_response = calibrated_frequency_response / (uncalibrated_frequency_response + epsilon)
    hamming_window = np.hamming(num_steps)
    windowed_response = calibrated_frequency_response * hamming_window
    # --- Step 4: Synthesize the A-scan (Range Profile) ---
    #a_scan = np.fft.ifftshift(np.fft.ifft(calibrated_frequency_response))
    a_scan = np.fft.ifft(windowed_response)

    return a_scan, calibrated_frequency_response, uncalibrated_frequency_response

def plotResults(ascan_profile, calibrated_frequency_response, echo_response):
    # --- Plot the result ---
    
    # Calculate the magnitude of the A-scan
    magnitude = np.abs(ascan_profile)**2
    
    # Normalize the magnitude so the peak is at 1.0 (or 0 dB)
    normalized_magnitude = magnitude / np.max(magnitude)
    
    # Convert to decibels (dB). Add a small epsilon to avoid log10(0).
    ascan_db = 10 * np.log10(normalized_magnitude + 1e-9)
    
    plt.figure(figsize=(10, 6))
    plt.plot(ascan_db)
    #plt.stem(ascan_db)
    plt.title("Generated A-Scan (Range Profile) in dB")
    plt.xlabel("Range Bin")
    plt.ylabel("Magnitude (dB)")
    plt.grid(True)
    plt.ylim(-60, 5) # Set a reasonable y-axis limit to see the noise floor
    
    peak_location = np.argmax(ascan_db)
    plt.axvline(x=peak_location, color='r', linestyle='--', label=f'Target Peak at bin {peak_location}')
    plt.legend()
    print(f"Generated A-scan. Target peak found at range bin: {peak_location}")
    
    plt.show()

    # Plot phase of compensated channel response
    plt.figure(figsize=(10, 6))
    plt.plot(np.angle(calibrated_frequency_response))
    plt.title("Phse of compensated frequency_response")
    plt.xlabel("Frequency steps")
    plt.ylabel("Phase")
    plt.grid(True)
    
    plt.show()

    # Plot phase of uncompensated channel response
    plt.figure(figsize=(10, 6))
    plt.plot(np.angle(echo_response))
    plt.title("Phse of uncompensated frequency_response")
    plt.xlabel("Frequency steps")
    plt.ylabel("Phase")
    plt.grid(True)
    
    plt.show()

def calculate_snr(a_scan: np.ndarray, exclusion_width: int = 3) -> float:
    """
    Calculates the Signal-to-Noise Ratio (SNR) of a radar A-scan.

    The SNR is calculated as the ratio of the peak signal power to the
    average noise power in the profile.

    Args:
        a_scan: A 1D numpy array containing the complex or real A-scan data.
        exclusion_width: The number of samples around the peak to exclude
                         from the noise calculation. Must be an odd number.

    Returns:
        The calculated SNR in decibels (dB).
    """
    if exclusion_width % 2 == 0:
        raise ValueError("Exclusion width must be an odd number.")

    # --- 1. Calculate Magnitudes and Find Peak Signal Power ---
    # Use np.abs() to handle both real and complex inputs
    magnitudes = np.abs(a_scan)
    signal_peak_magnitude = np.max(magnitudes)
    signal_power = signal_peak_magnitude**2
    peak_index = np.argmax(magnitudes)
    print("signal_power: ",signal_power)

    # --- 2. Identify and Calculate Average Noise Power ---
    # Create a copy of the magnitudes to use for noise calculation
    noise_magnitudes = magnitudes.copy()
    
    # Define the exclusion zone around the peak
    half_width = exclusion_width // 2
    start_exclusion = max(0, peak_index - half_width)
    end_exclusion = min(len(a_scan), peak_index + half_width + 1)
    
    # Set the values in the exclusion zone to NaN (Not a Number)
    noise_magnitudes[start_exclusion:end_exclusion] = np.nan
    
    # Calculate the average noise power using nanmean, which ignores NaNs
    average_noise_power = np.nanmean(noise_magnitudes**2)
    print("average_noise_power: ",average_noise_power)
    # --- 3. Calculate SNR in dB ---
    # Add a small epsilon to avoid log10(0) if noise power is zero
    epsilon = 1e-12
    snr_ratio = signal_power / (average_noise_power + epsilon)
    snr_db = 10 * np.log10(snr_ratio)

    return snr_db

def calculate_processing_gain(
    frequency_sweeps: np.ndarray,
    stack_sizes: list = None
) -> (list, list):
    """
    Calculates SNR improvement from coherent averaging (stacking).

    Args:
        frequency_sweeps: A 2D numpy array where each row is a complete,
                          calibrated complex frequency response vector (H-bar).
                          Shape: (num_sweeps, num_steps).
        stack_sizes: A list of integers representing the number of sweeps
                     to average (e.g., [2, 4, 8, 16, ...]). If None, it will
                     default to powers of 2 up to the number of sweeps.

    Returns:
        A tuple containing two lists:
        - The list of stack sizes (N) used.
        - The corresponding list of calculated processing gains in dB.
    """
    num_sweeps, num_steps = frequency_sweeps.shape

    if stack_sizes is None:
        # Default to powers of 2 up to the total number of sweeps
        max_power = int(np.log2(num_sweeps))
        stack_sizes = [2**i for i in range(1, max_power + 1)]

    # --- Step 1: Calculate Single-Pulse SNR ---
    first_sweep = frequency_sweeps[0, :]
    a_scan_single = np.fft.ifftshift(np.fft.ifft(first_sweep))
    snr_single = calculate_snr(a_scan_single)

    if snr_single == -np.inf:
        print("Warning: Single pulse SNR is zero. Cannot calculate gain.")
        return [], []

    print(f"Baseline Single-Pulse SNR: {snr_single:.2f} dB")

    # --- Step 2: Coherently Average and Calculate Gain for each Stack Size ---
    processing_gains_db = []
    actual_stack_sizes = []

    for n_stacks in stack_sizes:
        if n_stacks > num_sweeps:
            continue # Skip if we don't have enough sweeps
        
        actual_stack_sizes.append(n_stacks)
        
        # Coherently average the first 'n_stacks' sweeps
        stacked_freq_response = np.mean(frequency_sweeps[:n_stacks, :], axis=0)
        
        # Generate the stacked A-scan
        a_scan_stacked = np.fft.ifft(stacked_freq_response)
        
        # Calculate the SNR of the stacked result
        snr_stacked = calculate_snr(a_scan_stacked)
        
        # Calculate the processing gain
        gain = snr_stacked - snr_single
        processing_gains_db.append(gain)
        
        print(f"  - Stacks (N={n_stacks:4d}): Stacked SNR={snr_stacked:6.2f} dB, Gain={gain:6.2f} dB")

    return actual_stack_sizes, processing_gains_db, a_scan_stacked, stacked_freq_response

In [ ]:
def find_second_largest_index(arr: np.ndarray) -> int:
    """
    Finds the index of the second largest element in a NumPy array.

    Args:
        arr: A 1D NumPy array of numbers.

    Returns:
        The index of the second largest element.
        
    Raises:
        ValueError: If the array has fewer than two elements.
    """
    if arr.size < 2:
        raise ValueError("Array must contain at least two elements to find a second largest.")
        
    # argsort() returns the indices that would sort the array in ascending order.
    # The index of the largest element will be the last one (-1),
    # and the second largest will be the one before it (-2).
    sorted_indices = np.argsort(arr)
    
    return sorted_indices[-2]

def find_index(arr: np.ndarray, order) -> int:
    """
    Finds the index of the second largest element in a NumPy array.

    Args:
        arr: A 1D NumPy array of numbers.

    Returns:
        The index of the second largest element.
        
    Raises:
        ValueError: If the array has fewer than two elements.
    """
    if arr.size < 2:
        raise ValueError("Array must contain at least two elements to find a second largest.")
        
    # argsort() returns the indices that would sort the array in ascending order.
    # The index of the largest element will be the last one (-1),
    # and the second largest will be the one before it (-2).
    sorted_indices = np.argsort(arr)
    
    return sorted_indices[order]